In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
# ============================================================
# STEP 1 — CLEAN ENVIRONMENT CHECK
# ============================================================

from pathlib import Path
import os
import torch
import ultralytics

from ultralytics import YOLO

print("=" * 80)
print("STEP 1 — ENVIRONMENT CHECK")
print("=" * 80)

print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")

assert torch.cuda.is_available(), "CUDA GPU is not available."

print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"CUDA        : {torch.version.cuda}")

DATA_YAML = Path(
    "/kaggle/working/visdrone_final.yaml"
)

assert DATA_YAML.exists(), (
    f"Dataset YAML not found: {DATA_YAML}"
)

print(f"\nDataset YAML: {DATA_YAML}")

print("\nDataset YAML contents:")
print("-" * 80)

print(DATA_YAML.read_text())

print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 2 — VERIFY TRUE 10-CLASS YOLOv8n
# ============================================================

from pathlib import Path
from ultralytics import YOLO
import torch

SOURCE_YAML = Path("/kaggle/working/yolov8n_visdrone_10class.yaml")

assert SOURCE_YAML.exists(), f"YAML not found: {SOURCE_YAML}"

# Reconstruct the model from the saved YAML
model = YOLO(str(SOURCE_YAML))

# ------------------------------------------------------------
# Find the Detect head
# ------------------------------------------------------------
detect_head = None

for module in model.model.model:
    if module.__class__.__name__ == "Detect":
        detect_head = module
        break

assert detect_head is not None, "Detect head not found."

# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(
    p.numel() for p in model.model.parameters()
    if p.requires_grad
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------
print("=" * 80)
print("STEP 2 — TRUE 10-CLASS YOLOv8n VERIFICATION")
print("=" * 80)

print(f"YAML       : {SOURCE_YAML}")
print(f"Detect nc  : {detect_head.nc}")
print(f"Total params: {total_params:,}")
print(f"Trainable   : {trainable_params:,}")

print("\nClass names:")
for i, name in enumerate(model.names):
    print(f"  {i}: {name}")

# ------------------------------------------------------------
# Hard checks
# ------------------------------------------------------------
assert detect_head.nc == 10, \
    f"Expected 10 classes, found {detect_head.nc}"

assert len(model.names) == 10, \
    f"Expected 10 class names, found {len(model.names)}"

assert total_params == 3_012_798, \
    f"Expected 3,012,798 parameters, found {total_params:,}"

print("\n" + "=" * 80)
print("✅ STEP 2 PASSED")
print("=" * 80)
print("True 10-class YOLOv8n architecture verified.")

In [3]:
# ============================================================
# STEP 3 — DEFINE HYBRID NECK
# ============================================================

import torch
import torch.nn as nn

print("=" * 80)
print("STEP 3 — HYBRID NECK DEFINITION")
print("=" * 80)


class HybridNeckBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.channels = channels

        # Local spatial branch
        self.local = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=channels,
            bias=False
        )

        # Dilated context branch
        self.context = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=1,
            padding=2,
            dilation=2,
            groups=channels,
            bias=False
        )

        # Pointwise fusion
        self.fusion = nn.Conv2d(
            channels * 2,
            channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

    def forward(self, x):

        local = self.local(x)

        context = self.context(x)

        fused = torch.cat(
            [local, context],
            dim=1
        )

        fused = self.fusion(fused)

        return x + fused


# ------------------------------------------------------------
# Parameter verification
# ------------------------------------------------------------

for c in [64, 128, 256]:

    block = HybridNeckBlock(c)

    n = sum(
        p.numel()
        for p in block.parameters()
    )

    print(
        f"HybridNeckBlock({c}) = {n:,} parameters"
    )

total_hybrid = sum(
    sum(p.numel() for p in HybridNeckBlock(c).parameters())
    for c in [64, 128, 256]
)

print(
    f"\nTotal Hybrid Neck parameters: "
    f"{total_hybrid:,}"
)

assert total_hybrid == 180_096

print("\n✅ Hybrid Neck definition confirmed.")
print("   Total parameters: 180,096")

print("\n" + "=" * 80)
print("STEP 3 COMPLETE")
print("=" * 80)

STEP 3 — HYBRID NECK DEFINITION
HybridNeckBlock(64) = 9,344 parameters
HybridNeckBlock(128) = 35,072 parameters
HybridNeckBlock(256) = 135,680 parameters

Total Hybrid Neck parameters: 180,096

✅ Hybrid Neck definition confirmed.
   Total parameters: 180,096

STEP 3 COMPLETE


In [ ]:
# ============================================================
# STEP 4 — REGISTER HYBRID NECK WITH ULTRALYTICS
# ============================================================

import ultralytics.nn.tasks as tasks

print("=" * 80)
print("STEP 4 — REGISTERING HYBRID NECK")
print("=" * 80)

# Make the custom module visible to Ultralytics YAML parser
tasks.HybridNeckBlock = HybridNeckBlock

print(
    "Registered HybridNeckBlock in "
    "ultralytics.nn.tasks"
)

assert hasattr(
    tasks,
    "HybridNeckBlock"
)

print("✅ Registration successful.")

print("\n" + "=" * 80)
print("STEP 4 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 5 — FINAL CORRECT HYBRID NECK YAML
# ============================================================

from pathlib import Path
import copy
import yaml

SOURCE_YAML = Path(
    "/kaggle/working/yolov8n_visdrone_10class.yaml"
)

HYBRID_YAML = Path(
    "/kaggle/working/yolov8n_visdrone_hybridneck_10class.yaml"
)

cfg = yaml.safe_load(
    SOURCE_YAML.read_text()
)

# ------------------------------------------------------------
# IMPORTANT:
#
# These are BASE YOLOv8 YAML channel values.
# With YOLOv8n width_multiple=0.25:
#
#   256 -> 64
#   512 -> 128
#   1024 -> 256
#
# Therefore the actual runtime P3/P4/P5 channels are:
#
#   P3 = 64
#   P4 = 128
#   P5 = 256
#
# We insert HybridNeckBlock directly at these locations.
# ------------------------------------------------------------

insert_after = {
    4: 64,       # actual P3
    6: 128,      # actual P4
    8: 256,      # actual P5
}

insert_indices = sorted(insert_after.keys())


# ------------------------------------------------------------
# Shift positive layer references
# ------------------------------------------------------------

def shift_ref(value):

    if isinstance(value, int) and value >= 0:
        return value + sum(
            insertion_idx < value
            for insertion_idx in insert_indices
        )

    return value


def shift_from(value):

    if isinstance(value, list):
        return [
            shift_ref(v)
            for v in value
        ]

    return shift_ref(value)


# ------------------------------------------------------------
# Rebuild backbone
# ------------------------------------------------------------

original_backbone = copy.deepcopy(
    cfg["backbone"]
)

new_backbone = []

for original_idx, layer in enumerate(
    original_backbone
):

    layer = copy.deepcopy(layer)

    # Shift references in original backbone
    layer[0] = shift_from(layer[0])

    new_backbone.append(layer)

    # Insert HybridNeckBlock
    if original_idx in insert_after:

        channels = insert_after[original_idx]

        new_backbone.append(
            [
                -1,
                1,
                "HybridNeckBlock",
                [channels]
            ]
        )


# ------------------------------------------------------------
# Rebuild head
# ------------------------------------------------------------

original_head = copy.deepcopy(
    cfg["head"]
)

new_head = []

for layer in original_head:

    layer = copy.deepcopy(layer)

    # Shift original head references exactly once
    layer[0] = shift_from(layer[0])

    new_head.append(layer)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

cfg["backbone"] = new_backbone
cfg["head"] = new_head

HYBRID_YAML.write_text(
    yaml.safe_dump(
        cfg,
        sort_keys=False
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("STEP 5 — FINAL CORRECT HYBRID NECK YAML")
print("=" * 80)

print(f"Saved: {HYBRID_YAML}")

print("\nHybrid Backbone:")

for i, layer in enumerate(new_backbone):
    print(f"{i:2d}: {layer}")

print("\nHybrid Head:")

for i, layer in enumerate(new_head):
    print(f"{i:2d}: {layer}")


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

hybrid_layers = [
    layer
    for layer in new_backbone
    if layer[2] == "HybridNeckBlock"
]

assert len(hybrid_layers) == 3

assert hybrid_layers[0][3] == [64]
assert hybrid_layers[1][3] == [128]
assert hybrid_layers[2][3] == [256]

# Original Detect references are [15, 18, 21].
# Three insertions after 4, 6, 8 produce:
# [18, 21, 24]

detect_layers = [
    layer
    for layer in new_head
    if layer[2] == "Detect"
]

assert len(detect_layers) == 1

assert detect_layers[0][0] == [18, 21, 24]

print("\nActual YOLOv8n runtime channels:")
print("  P3 = 64")
print("  P4 = 128")
print("  P5 = 256")

print("\nHybrid Neck:")
print("  P3 = HybridNeckBlock(64)")
print("  P4 = HybridNeckBlock(128)")
print("  P5 = HybridNeckBlock(256)")

print("\nExpected Hybrid Neck parameters: 180,096")
print("Expected total model parameters:  3,192,894")

print("\nDetect references: [18, 21, 24]")

print("\n" + "=" * 80)
print("✅ STEP 5 CORRECTED AND COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 6 — RECONSTRUCT AND VERIFY HYBRID NECK MODEL
# ============================================================

from ultralytics import YOLO
import torch

HYBRID_YAML = "/kaggle/working/yolov8n_visdrone_hybridneck_10class.yaml"

print("=" * 80)
print("STEP 6 — HYBRID NECK MODEL RECONSTRUCTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Reconstruct directly from YAML
# ------------------------------------------------------------

print("\nLoading Hybrid Neck model...")

hybrid_model = YOLO(HYBRID_YAML)

print("✅ Model reconstructed successfully.")


# ------------------------------------------------------------
# 2. Inspect HybridNeckBlock modules
# ------------------------------------------------------------

hybrid_modules = []

for idx, module in enumerate(hybrid_model.model.model):

    if isinstance(module, HybridNeckBlock):
        hybrid_modules.append((idx, module))


print("\nDetected HybridNeckBlock modules:")

for idx, module in hybrid_modules:
    print(
        f"  Layer {idx}: "
        f"channels={module.channels}"
    )


# ------------------------------------------------------------
# 3. Verify exactly three modules
# ------------------------------------------------------------

assert len(hybrid_modules) == 3, (
    f"Expected 3 HybridNeckBlock modules, "
    f"found {len(hybrid_modules)}"
)

expected_channels = [64, 128, 256]

actual_channels = [
    module.channels
    for _, module in hybrid_modules
]

assert actual_channels == expected_channels, (
    f"Unexpected Hybrid Neck channels: {actual_channels}"
)

print("\n✅ Hybrid module count: 3")
print("✅ Hybrid channel configuration: [64, 128, 256]")


# ------------------------------------------------------------
# 4. Count parameters
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in hybrid_model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in hybrid_model.model.parameters()
    if p.requires_grad
)

frozen_params = (
    total_params - trainable_params
)

print("\nParameter count:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {frozen_params:,}")


# ------------------------------------------------------------
# 5. Verify total parameter count
# ------------------------------------------------------------

EXPECTED_TOTAL = 3_192_894

assert total_params == EXPECTED_TOTAL, (
    f"Parameter mismatch: expected "
    f"{EXPECTED_TOTAL:,}, got {total_params:,}"
)

print("\n✅ Total parameter count confirmed.")


# ------------------------------------------------------------
# 6. Verify individual HybridNeckBlock parameters
# ------------------------------------------------------------

expected_block_params = [
    9_344,
    35_072,
    135_680,
]

print("\nIndividual HybridNeckBlock parameters:")

for (idx, module), expected in zip(
    hybrid_modules,
    expected_block_params
):

    actual = sum(
        p.numel()
        for p in module.parameters()
    )

    print(
        f"  Layer {idx}: "
        f"{actual:,} parameters"
    )

    assert actual == expected


print("✅ Individual block parameter counts confirmed.")


# ------------------------------------------------------------
# 7. Verify Detect layer
# ------------------------------------------------------------

detect_layers = []

for idx, module in enumerate(
    hybrid_model.model.model
):

    if module.__class__.__name__ == "Detect":
        detect_layers.append((idx, module))


assert len(detect_layers) == 1

detect_idx, detect_module = detect_layers[0]

print("\nDetect layer:")
print(f"  Layer index: {detect_idx}")
print(f"  Number of classes: {detect_module.nc}")

assert detect_module.nc == 10

print("✅ Detect head configured for 10 classes.")


# ------------------------------------------------------------
# 8. Dummy forward pass
# ------------------------------------------------------------

print("\nRunning dummy forward pass...")

device = next(
    hybrid_model.model.parameters()
).device

dummy = torch.zeros(
    1,
    3,
    640,
    640,
    device=device
)

with torch.no_grad():
    output = hybrid_model.model(dummy)

print("✅ Dummy forward pass successful.")


# ------------------------------------------------------------
# 9. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 6 VERIFICATION SUMMARY")
print("=" * 80)

print("Model reconstruction:       PASS")
print("Hybrid modules:             PASS")
print("Channels [64,128,256]:      PASS")
print("Block parameters:           PASS")
print("Total parameters:           PASS")
print("10-class Detect head:       PASS")
print("Dummy forward:              PASS")

print("\nExpected architecture:")
print("  P3 -> HybridNeckBlock(64)")
print("  P4 -> HybridNeckBlock(128)")
print("  P5 -> HybridNeckBlock(256)")

print("\nExpected total parameters: 3,192,894")

print("\n" + "=" * 80)
print("✅ STEP 6 COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

print("=" * 80)
print("STEP 7 — COMPATIBLE PRETRAINED WEIGHT TRANSFER")
print("=" * 80)

HYBRID_YAML = "/kaggle/working/yolov8n_visdrone_hybridneck_10class.yaml"
COCO_WEIGHTS = "/kaggle/working/yolov8n.pt"

# ------------------------------------------------------------------
# 1. Load the validated Hybrid Neck architecture
# ------------------------------------------------------------------
print("\nLoading validated Hybrid Neck architecture...")
hybrid_model = YOLO(HYBRID_YAML)

# ------------------------------------------------------------------
# 2. Load official COCO YOLOv8n checkpoint
# ------------------------------------------------------------------
print("Loading YOLOv8n COCO checkpoint...")
coco_model = YOLO(COCO_WEIGHTS)

src_state = coco_model.model.state_dict()
dst_state = hybrid_model.model.state_dict()

print(f"\nSource checkpoint tensors: {len(src_state)}")
print(f"Target model tensors:      {len(dst_state)}")

# ------------------------------------------------------------------
# 3. Transfer only exact key + shape matches
# ------------------------------------------------------------------
compatible = {}
skipped_shape = []
skipped_missing = []

for key, src_tensor in src_state.items():

    if key not in dst_state:
        skipped_missing.append(key)
        continue

    dst_tensor = dst_state[key]

    if src_tensor.shape == dst_tensor.shape:
        compatible[key] = src_tensor.clone()
    else:
        skipped_shape.append(
            (key, tuple(src_tensor.shape), tuple(dst_tensor.shape))
        )

print("\nCompatible tensors found:", len(compatible))
print("Skipped — missing keys:", len(skipped_missing))
print("Skipped — shape mismatch:", len(skipped_shape))

# ------------------------------------------------------------------
# 4. Protect HybridNeckBlock weights
# ------------------------------------------------------------------
hybrid_keys = [
    key
    for key in dst_state.keys()
    if "HybridNeckBlock" in key
]

print("\nHybridNeckBlock tensors:")
for key in hybrid_keys:
    print(" ", key)

# These must NOT come from the COCO checkpoint.
overlap = set(hybrid_keys) & set(compatible.keys())

print("\nHybrid tensors transferred from COCO:", len(overlap))

assert len(overlap) == 0, (
    "ERROR: HybridNeckBlock parameters were included in pretrained transfer."
)

# ------------------------------------------------------------------
# 5. Load compatible weights
# ------------------------------------------------------------------
load_result = hybrid_model.model.load_state_dict(
    compatible,
    strict=False
)

print("\nLoad result:")
print("  Missing keys:", len(load_result.missing_keys))
print("  Unexpected keys:", len(load_result.unexpected_keys))

# ------------------------------------------------------------------
# 6. Verify Hybrid Neck remains present
# ------------------------------------------------------------------
hybrid_modules = []

for idx, module in enumerate(hybrid_model.model.model):
    if isinstance(module, HybridNeckBlock):
        hybrid_modules.append((idx, module))

assert len(hybrid_modules) == 3
assert [m.channels for _, m in hybrid_modules] == [64, 128, 256]

# ------------------------------------------------------------------
# 7. Verify Detect head
# ------------------------------------------------------------------
detect_modules = [
    (idx, module)
    for idx, module in enumerate(hybrid_model.model.model)
    if module.__class__.__name__ == "Detect"
]

assert len(detect_modules) == 1
detect_idx, detect_module = detect_modules[0]

assert detect_module.nc == 10

# ------------------------------------------------------------------
# 8. Parameter verification
# ------------------------------------------------------------------
total_params = sum(
    p.numel() for p in hybrid_model.model.parameters()
)

assert total_params == 3_192_894

print("\nArchitecture verification:")
print("  Hybrid modules:      PASS")
print("  Hybrid channels:     PASS")
print("  Total parameters:     PASS")
print("  Detect classes:       PASS")

# ------------------------------------------------------------------
# 9. Dummy forward
# ------------------------------------------------------------------
print("\nRunning dummy forward after weight transfer...")

device = next(hybrid_model.model.parameters()).device
dummy = torch.zeros(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = hybrid_model.model(dummy)

print("✅ Dummy forward after weight transfer successful.")

# ------------------------------------------------------------------
# 10. Summary
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 7 VERIFICATION SUMMARY")
print("=" * 80)

print(f"Compatible pretrained tensors transferred: {len(compatible)}")
print(f"HybridNeckBlock tensors protected:         {len(hybrid_keys)}")
print(f"Hybrid tensors accidentally transferred:   {len(overlap)}")
print(f"Total model parameters:                     {total_params:,}")
print(f"Detect classes:                             {detect_module.nc}")

print("\nExpected:")
print("  HybridNeckBlock weights: RANDOM INITIALIZATION")
print("  Compatible YOLOv8n weights: TRANSFERRED")
print("  Detection head: 10-class configuration")
print("  Architecture: PRESERVED")

print("\n" + "=" * 80)
print("✅ STEP 7 COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("STEP 8 — PRETRAINED WEIGHT TRANSFER AUDIT")
print("=" * 80)

# Reconstruct the source and target state dictionaries
src_state = coco_model.model.state_dict()
dst_state = hybrid_model.model.state_dict()

transferred = []

for key, src_tensor in src_state.items():
    if key in dst_state and src_tensor.shape == dst_state[key].shape:
        transferred.append(
            (key, tuple(src_tensor.shape), src_tensor.numel())
        )

print(f"\nTotal transferred tensors: {len(transferred)}")
print(f"Total transferred parameters: {sum(x[2] for x in transferred):,}")

print("\nTransferred tensors:")
for key, shape, numel in transferred:
    print(f"  {key:45s} {str(shape):25s} {numel:,}")

# ---------------------------------------------------------------
# Verify that no HybridNeckBlock parameter was transferred
# ---------------------------------------------------------------
hybrid_keys = [
    key for key in dst_state.keys()
    if "HybridNeckBlock" in key
]

transferred_hybrid = [
    key for key, _, _ in transferred
    if "HybridNeckBlock" in key
]

print("\nHybridNeckBlock tensors in target:")
print(f"  {len(hybrid_keys)}")

print("HybridNeckBlock tensors transferred:")
print(f"  {len(transferred_hybrid)}")

assert len(transferred_hybrid) == 0

# ---------------------------------------------------------------
# Identify Detect-head tensors
# ---------------------------------------------------------------
detect_keys = [
    key for key in dst_state.keys()
    if ".cv2." in key or ".cv3." in key
]

transferred_detect = [
    key for key, _, _ in transferred
    if key in detect_keys
]

print("\nDetection-head tensors in target:")
print(f"  {len(detect_keys)}")

print("Detection-head tensors transferred:")
print(f"  {len(transferred_detect)}")

# ---------------------------------------------------------------
# Final audit
# ---------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 8 VERIFICATION SUMMARY")
print("=" * 80)

print(f"Transferred tensors:              {len(transferred)}")
print(f"Transferred parameters:           {sum(x[2] for x in transferred):,}")
print(f"Hybrid tensors transferred:       {len(transferred_hybrid)}")
print(f"Detection tensors transferred:    {len(transferred_detect)}")

assert len(transferred) == 97
assert len(transferred_hybrid) == 0

print("\n✅ STEP 8 COMPLETE")

In [ ]:
from ultralytics import YOLO
import torch

print("=" * 80)
print("STEP 9 — SEMANTIC PRETRAINED WEIGHT MAPPING")
print("=" * 80)

HYBRID_YAML = "/kaggle/working/yolov8n_visdrone_hybridneck_10class.yaml"
COCO_WEIGHTS = "/kaggle/working/yolov8n.pt"

# Fresh models — do NOT reuse the Step 7 transferred model
print("\nLoading fresh Hybrid Neck model...")
fresh_hybrid = YOLO(HYBRID_YAML)

print("Loading original YOLOv8n COCO model...")
coco = YOLO(COCO_WEIGHTS)

src = coco.model.model
dst = fresh_hybrid.model.model

# Explicit semantic mapping:
# source YOLOv8n layer -> target Hybrid-Neck layer
layer_mapping = {
    0: 0,    # Conv
    1: 1,    # Conv
    2: 2,    # C2f
    3: 3,    # Conv
    4: 4,    # C2f
    5: 6,    # Conv
    6: 7,    # C2f
    7: 9,    # Conv
    8: 10,   # C2f
    9: 12,   # SPPF
}

print("\nExplicit backbone mapping:")
print("-" * 80)

compatible_layers = []
incompatible_layers = []

for src_idx, dst_idx in layer_mapping.items():

    src_module = src[src_idx]
    dst_module = dst[dst_idx]

    src_state = src_module.state_dict()
    dst_state = dst_module.state_dict()

    matched = []
    mismatched = []

    for key, src_tensor in src_state.items():

        if key not in dst_state:
            mismatched.append(
                (key, "missing target key")
            )
            continue

        if src_tensor.shape != dst_state[key].shape:
            mismatched.append(
                (
                    key,
                    f"{tuple(src_tensor.shape)} -> "
                    f"{tuple(dst_state[key].shape)}"
                )
            )
        else:
            matched.append(key)

    print(
        f"\nSource layer {src_idx:2d} "
        f"({src_module.__class__.__name__:<12})"
        f" -> Target layer {dst_idx:2d} "
        f"({dst_module.__class__.__name__:<12})"
    )

    print(f"  Compatible tensors: {len(matched)}")
    print(f"  Incompatible tensors: {len(mismatched)}")

    if mismatched:
        for item in mismatched:
            print("   ", item)

        incompatible_layers.append((src_idx, dst_idx))
    else:
        compatible_layers.append((src_idx, dst_idx))

print("\n" + "=" * 80)
print("STEP 9 VERIFICATION SUMMARY")
print("=" * 80)

print(f"Mapped backbone layers:        {len(layer_mapping)}")
print(f"Fully compatible layer pairs:  {len(compatible_layers)}")
print(f"Incompatible layer pairs:      {len(incompatible_layers)}")

print("\nHybrid layers intentionally excluded:")
print("  Target layer 5  -> HybridNeckBlock(64)")
print("  Target layer 8  -> HybridNeckBlock(128)")
print("  Target layer 11 -> HybridNeckBlock(256)")

print("\nDetection head:")
print("  COCO detection-head transfer: EXCLUDED")
print("  Target classes: 10")

assert len(incompatible_layers) == 0
assert len(compatible_layers) == len(layer_mapping)

print("\n✅ Semantic mapping verified.")
print("✅ No HybridNeckBlock weight transfer.")
print("✅ Detection head excluded.")
print("✅ Ready for controlled weight transfer.")

In [ ]:
print("=" * 80)
print("STEP 10 — APPLY SEMANTIC PRETRAINED WEIGHT TRANSFER")
print("=" * 80)

# fresh_hybrid and coco are already loaded from Step 9
src = coco.model.model
dst = fresh_hybrid.model.model

# Explicit semantic mapping verified in Step 9
layer_mapping = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 6,
    6: 7,
    7: 9,
    8: 10,
    9: 12,
}

transferred_tensors = 0
transferred_parameters = 0

print("\nTransferring compatible backbone weights...")

for src_idx, dst_idx in layer_mapping.items():

    src_state = src[src_idx].state_dict()
    dst_state = dst[dst_idx].state_dict()

    updated_state = {}

    for key, src_tensor in src_state.items():

        assert key in dst_state
        assert src_tensor.shape == dst_state[key].shape

        updated_state[key] = src_tensor.clone()

        transferred_tensors += 1
        transferred_parameters += src_tensor.numel()

    dst[dst_idx].load_state_dict(
        updated_state,
        strict=True
    )

    print(
        f"  Source {src_idx:2d} -> Target {dst_idx:2d} | "
        f"{len(updated_state):2d} tensors | "
        f"{sum(v.numel() for v in updated_state.values()):,} parameters"
    )

# ------------------------------------------------------------------
# Verify HybridNeckBlock remains untouched
# ------------------------------------------------------------------
hybrid_layers = [5, 8, 11]

print("\nHybrid Neck layers:")
for idx in hybrid_layers:
    module = dst[idx]

    assert isinstance(module, HybridNeckBlock)

    n_params = sum(
        p.numel()
        for p in module.parameters()
    )

    print(
        f"  Layer {idx}: "
        f"HybridNeckBlock({module.channels}) | "
        f"{n_params:,} parameters | "
        f"RANDOM INITIALIZATION PRESERVED"
    )

# ------------------------------------------------------------------
# Verify detection head
# ------------------------------------------------------------------
detect_idx = 25
detect_module = dst[detect_idx]

assert detect_module.__class__.__name__ == "Detect"
assert detect_module.nc == 10

print("\nDetection head:")
print(f"  Layer: {detect_idx}")
print(f"  Classes: {detect_module.nc}")
print("  COCO detection weights: NOT transferred")
print("  VisDrone 10-class head: preserved")

# ------------------------------------------------------------------
# Verify total architecture
# ------------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in fresh_hybrid.model.parameters()
)

assert total_params == 3_192_894

# ------------------------------------------------------------------
# Dummy forward
# ------------------------------------------------------------------
print("\nRunning dummy forward...")

device = next(fresh_hybrid.model.parameters()).device

dummy = torch.zeros(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = fresh_hybrid.model(dummy)

print("✅ Dummy forward successful.")

# ------------------------------------------------------------------
# Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 10 VERIFICATION SUMMARY")
print("=" * 80)

print(f"Backbone tensors transferred:      {transferred_tensors}")
print(f"Backbone parameters transferred:   {transferred_parameters:,}")
print("HybridNeckBlock parameters:        RANDOM INITIALIZATION")
print("Detection head:                    10-class VisDrone initialization")
print(f"Total model parameters:            {total_params:,}")
print("Dummy forward:                     PASS")

print("\n" + "=" * 80)
print("✅ STEP 10 COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import torch

print("=" * 80)
print("STEP 11 — SAVE INITIALIZED HYBRID NECK MODEL")
print("=" * 80)

SAVE_PATH = Path(
    "/kaggle/working/yolov8n_hybridneck_visdrone_10class_initialized.pt"
)

# Save the complete Ultralytics model object.
torch.save(
    fresh_hybrid.model,
    SAVE_PATH
)

assert SAVE_PATH.exists()
assert SAVE_PATH.stat().st_size > 0

print("\nSaved initialized model:")
print(f"  Path: {SAVE_PATH}")
print(f"  Size: {SAVE_PATH.stat().st_size / (1024**2):.2f} MB")

# ---------------------------------------------------------------
# Verify saved object can be loaded
# ---------------------------------------------------------------
print("\nReloading saved model for verification...")

reloaded = torch.load(
    SAVE_PATH,
    map_location="cpu",
    weights_only=False
)

print("✅ Saved model reloaded successfully.")

# ---------------------------------------------------------------
# Verify architecture after serialization
# ---------------------------------------------------------------
hybrid_modules = []

for idx, module in enumerate(reloaded.model):
    if isinstance(module, HybridNeckBlock):
        hybrid_modules.append((idx, module))

assert len(hybrid_modules) == 3
assert [m.channels for _, m in hybrid_modules] == [64, 128, 256]

total_params = sum(
    p.numel()
    for p in reloaded.parameters()
)

assert total_params == 3_192_894

detect_modules = [
    module
    for module in reloaded.model
    if module.__class__.__name__ == "Detect"
]

assert len(detect_modules) == 1
assert detect_modules[0].nc == 10

print("\nReloaded architecture:")
print("  Hybrid modules: PASS")
print("  Channels: [64, 128, 256] PASS")
print(f"  Total parameters: {total_params:,} PASS")
print("  Detect classes: 10 PASS")

print("\n" + "=" * 80)
print("STEP 11 VERIFICATION SUMMARY")
print("=" * 80)

print("Initialized model saved:          PASS")
print("Serialization/reload:             PASS")
print("Hybrid Neck preserved:            PASS")
print("10-class Detect preserved:        PASS")
print(f"Total parameters:                 {total_params:,}")

print("\n" + "=" * 80)
print("✅ STEP 11 COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("=" * 80)
print("STEP 11B — CORRECT ULTRALYTICS MODEL SERIALIZATION")
print("=" * 80)

# fresh_hybrid is the correctly initialized model from Step 10.
assert isinstance(fresh_hybrid, YOLO)

CORRECT_PATH = "/kaggle/working/yolov8n_hybridneck_visdrone_10class_initialized_ultralytics.pt"

print("\nSaving using Ultralytics YOLO.save()...")
fresh_hybrid.save(CORRECT_PATH)

assert Path(CORRECT_PATH).exists()
assert Path(CORRECT_PATH).stat().st_size > 0

print(f"Saved: {CORRECT_PATH}")
print(
    f"Size: {Path(CORRECT_PATH).stat().st_size / (1024**2):.2f} MB"
)

# ------------------------------------------------------------------
# Reload using YOLO()
# ------------------------------------------------------------------
print("\nReloading using YOLO()...")

train_model = YOLO(CORRECT_PATH)

print("✅ Ultralytics checkpoint reloaded successfully.")

# ------------------------------------------------------------------
# Verify actual YOLO model structure
# ------------------------------------------------------------------
assert hasattr(train_model, "model")
assert hasattr(train_model.model, "model")

hybrid_modules = []

for idx, module in enumerate(train_model.model.model):
    if isinstance(module, HybridNeckBlock):
        hybrid_modules.append((idx, module))

print("\nHybridNeckBlock modules:")

for idx, module in hybrid_modules:
    print(
        f"  Layer {idx}: "
        f"HybridNeckBlock({module.channels})"
    )

assert len(hybrid_modules) == 3
assert [m.channels for _, m in hybrid_modules] == [64, 128, 256]

# ------------------------------------------------------------------
# Verify parameter count
# ------------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in train_model.model.parameters()
)

assert total_params == 3_192_894

# ------------------------------------------------------------------
# Verify Detect head
# ------------------------------------------------------------------
detect_modules = [
    (idx, module)
    for idx, module in enumerate(train_model.model.model)
    if module.__class__.__name__ == "Detect"
]

assert len(detect_modules) == 1

detect_idx, detect_module = detect_modules[0]

assert detect_module.nc == 10

# ------------------------------------------------------------------
# Dummy forward
# ------------------------------------------------------------------
print("\nRunning dummy forward...")

import torch

device = next(train_model.model.parameters()).device

dummy = torch.zeros(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = train_model.model(dummy)

print("✅ Dummy forward successful.")

# ------------------------------------------------------------------
# Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 11B VERIFICATION SUMMARY")
print("=" * 80)

print("Ultralytics save():              PASS")
print("Ultralytics reload:              PASS")
print("HybridNeckBlock count:           3")
print("Hybrid channels:                 [64, 128, 256]")
print(f"Total parameters:                {total_params:,}")
print(f"Detect layer:                    {detect_idx}")
print("Detect classes:                  10")
print("Dummy forward:                   PASS")

print("\n" + "=" * 80)
print("✅ STEP 11B COMPLETE")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("=" * 80)
print("STEP 12B — TRAINING RECONSTRUCTION SMOKE TEST")
print("=" * 80)

MODEL_PATH = (
    "/kaggle/working/"
    "yolov8n_hybridneck_visdrone_10class_initialized_ultralytics.pt"
)

DATA_YAML = "/kaggle/working/visdrone_final.yaml"

SMOKE_PROJECT = "/kaggle/working/YOLO_VisDrone_SmokeTest"
SMOKE_NAME = "HybridNeck_train_reconstruction"

assert Path(MODEL_PATH).exists()
assert Path(DATA_YAML).exists()

print("\nLoading initialized Ultralytics checkpoint...")
smoke_model = YOLO(MODEL_PATH)

print("✅ Checkpoint loaded.")

# ---------------------------------------------------------------
# Verify custom module before train()
# ---------------------------------------------------------------
hybrid_before = [
    (idx, module)
    for idx, module in enumerate(smoke_model.model.model)
    if isinstance(module, HybridNeckBlock)
]

assert len(hybrid_before) == 3
assert [m.channels for _, m in hybrid_before] == [64, 128, 256]

print("\nBefore train():")
print("  Hybrid layers: 5, 8, 11")
print("  Channels: [64, 128, 256]")

# ---------------------------------------------------------------
# ONE-EPOCH SMOKE TEST
# ---------------------------------------------------------------
print("\nStarting 1-epoch training reconstruction test...")

results = smoke_model.train(
    data=DATA_YAML,

    # Minimal smoke test
    epochs=1,
    imgsz=640,
    batch=2,

    device=0,

    # Same optimizer family as the real experiment
    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=3,

    # Reproducibility
    seed=42,
    deterministic=True,

    # Runtime
    workers=2,
    amp=True,

    # Required detection settings
    max_det=1000,

    # IMPORTANT:
    # This is only a reconstruction test.
    # Do NOT use Copy-Paste yet.
    copy_paste=0.0,

    # No external pretrained replacement
    pretrained=False,

    # Smoke-test output
    project=SMOKE_PROJECT,
    name=SMOKE_NAME,
    exist_ok=True,

    # Avoid unnecessary early stopping during 1 epoch
    patience=1,
)

print("\n✅ train() returned successfully.")

# ---------------------------------------------------------------
# Verify model AFTER train()
# ---------------------------------------------------------------
hybrid_after = [
    (idx, module)
    for idx, module in enumerate(smoke_model.model.model)
    if isinstance(module, HybridNeckBlock)
]

print("\nAfter train():")
for idx, module in hybrid_after:
    print(
        f"  Layer {idx}: "
        f"HybridNeckBlock({module.channels})"
    )

assert len(hybrid_after) == 3
assert [m.channels for _, m in hybrid_after] == [64, 128, 256]

# ---------------------------------------------------------------
# Verify Detect head
# ---------------------------------------------------------------
detect_modules = [
    (idx, module)
    for idx, module in enumerate(smoke_model.model.model)
    if module.__class__.__name__ == "Detect"
]

assert len(detect_modules) == 1
detect_idx, detect_module = detect_modules[0]

assert detect_module.nc == 10

# ---------------------------------------------------------------
# Verify parameter count
# ---------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in smoke_model.model.parameters()
)

assert total_params == 3_192_894

print("\nPost-training architecture:")
print("  Hybrid Neck:       PASS")
print("  Channels:          [64, 128, 256]")
print(f"  Total parameters:  {total_params:,}")
print(f"  Detect layer:      {detect_idx}")
print("  Detect classes:     10")

print("\n" + "=" * 80)
print("STEP 12B VERIFICATION SUMMARY")
print("=" * 80)

print("train() reconstruction:       PASS")
print("Hybrid Neck survived train(): PASS")
print("10-class Detect:              PASS")
print(f"Total parameters:             {total_params:,}")
print("Copy-Paste:                   NOT USED")
print("Smoke-test purpose:           ARCHITECTURE ONLY")

print("\n" + "=" * 80)
print("✅ STEP 12B COMPLETE")
print("=" * 80)

In [ ]:
copy_paste=0.1

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ============================================================
# YOLOv8n + Hybrid Neck + Copy-Paste (0.1)
# Controlled experiment — Seed 42
# ============================================================

MODEL_PATH = "/kaggle/working/yolov8n_hybridneck_visdrone_10class_initialized_ultralytics.pt"
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

PROJECT_DIR = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv8n_HybridNeck_CopyPaste_AdamW_seed42"

# ------------------------------------------------------------
# Load correctly serialized Ultralytics checkpoint
# ------------------------------------------------------------
model = YOLO(MODEL_PATH)

# ------------------------------------------------------------
# Verify architecture BEFORE training
# ------------------------------------------------------------
print("=" * 80)
print("PRE-TRAINING ARCHITECTURE VERIFICATION")
print("=" * 80)

hybrid_layers = []

for i, layer in enumerate(model.model.model):
    if layer.__class__.__name__ == "HybridNeckBlock":
        hybrid_layers.append((i, layer.channels))
        print(f"Layer {i}: HybridNeckBlock({layer.channels})")

assert len(hybrid_layers) == 3, \
    f"Expected 3 HybridNeckBlock layers, found {len(hybrid_layers)}"

assert [c for _, c in hybrid_layers] == [64, 128, 256], \
    f"Unexpected Hybrid Neck channels: {hybrid_layers}"

detect_layers = [
    layer for layer in model.model.model
    if layer.__class__.__name__ == "Detect"
]

assert len(detect_layers) == 1, \
    f"Expected 1 Detect layer, found {len(detect_layers)}"

assert detect_layers[0].nc == 10, \
    f"Expected 10 classes, found {detect_layers[0].nc}"

print()
print("Hybrid Neck:       PASS")
print("Channels:          [64, 128, 256]")
print("10-class Detect:   PASS")
print(f"Total parameters:  {sum(p.numel() for p in model.model.parameters()):,}")

print("=" * 80)
print("STARTING 100-EPOCH CONTROLLED EXPERIMENT")
print("=" * 80)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
results = model.train(
    data=DATA_YAML,

    # Training duration
    epochs=100,

    # Input / hardware
    imgsz=640,
    batch=8,
    device=0,
    workers=2,

    # Optimizer
    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,

    # LR schedule
    cos_lr=True,
    warmup_epochs=3,

    # Reproducibility
    seed=42,
    deterministic=True,

    # Training behavior
    amp=True,
    patience=20,
    max_det=1000,

    # ONLY intentional change from Hybrid-Neck-only experiment
    copy_paste=0.1,

    # Important: use the initialized Hybrid-Neck checkpoint
    pretrained=False,

    # Output
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
)

print()
print("=" * 80)
print("TRAINING COMPLETED")
print("=" * 80)

# ------------------------------------------------------------
# Post-training architecture verification
# ------------------------------------------------------------
trained_model = YOLO(
    f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
)

hybrid_layers_after = []

for i, layer in enumerate(trained_model.model.model):
    if layer.__class__.__name__ == "HybridNeckBlock":
        hybrid_layers_after.append((i, layer.channels))

detect_layers_after = [
    layer for layer in trained_model.model.model
    if layer.__class__.__name__ == "Detect"
]

print()
print("POST-TRAINING ARCHITECTURE VERIFICATION")
print("-" * 80)

print("Hybrid Neck layers:", hybrid_layers_after)
print("Detect classes:", detect_layers_after[0].nc)
print(
    "Total parameters:",
    f"{sum(p.numel() for p in trained_model.model.parameters()):,}"
)

assert hybrid_layers_after == [
    (5, 64),
    (8, 128),
    (11, 256)
], f"Hybrid Neck changed after training: {hybrid_layers_after}"

assert detect_layers_after[0].nc == 10

print()
print("Hybrid Neck survived training: PASS")
print("10-class Detect head:           PASS")
print("Copy-Paste:                     0.1")
print("=" * 80)

In [4]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv8n_HybridNeck_CopyPaste": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_HybridNeck_CopyPaste_AdamW_seed42/weights/best.pt",
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n_visdrone_hybridneck_10class summary (fused): 81 layers, 3,187,694 parameters, 0 gradients, 8.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 5.9±2.3 MB/s, size: 157.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 40.1it/s 40.1s0.1ss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 3.4it/s 29.7s0.2s
                   all       1610      75102       0.35      0.284      0.238      0.129
 

In [5]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv8n_HybridNeck_CopyPaste=/kaggle/working/runs/detect/YOLOv8n_HybridNeck_CopyPaste/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv8n_HybridNeck_CopyPaste
Prediction file: /kaggle/working/runs/detect/YOLOv8n_HybridNeck_CopyPaste/predictions.json
Loading and preparing results...
DONE (t=3.51s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=195.65s).
Accumulating evaluation results...
DONE (t=13.16s).
YOLOv8n_HybridNeck_CopyPaste all     n= 75102 AP50=0.240 AP50-95=0.132 AR50=0.544 AR50-95=0.293
YOLOv8n_HybridNeck_CopyPaste tiny    n= 27346 AP50=0.068 AP50-95=0.023 AR50=0.258 AR50-95=0.096
YOLOv8n_HybridNeck_CopyPaste small   n= 23491 AP50=0.189 AP50-95=0.079 AR50=0.556 AR50-95=0.252
YOLOv8n_HybridNeck_CopyPaste medium  n= 21856 AP50=0.361 AP50-95=0.201 AR50=0.747 AR50-95=0.428
YOLOv8n_HybridNeck_CopyPaste large   n=  2409 AP50=0.484 AP50-95=0.330 AR50=0.812 AR50-95=0.572

Saved:
/kaggle/working/size_results.csv
